# GNN Robustness Reproducibility Notebook

Executable notebook for the final project: environment information, Cora loading, GCN training demonstration, final perturbations, raw-result aggregation, confidence intervals, and a real PCA embedding from hidden GCN representations.


In [ ]:
from pathlib import Path
import json
import platform
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

{"python": platform.python_version(), "torch": torch.__version__, "root": str(ROOT)}


## Dataset Loading


In [ ]:
from gnn_robustness.data import load_planetoid, cora_summary

dataset, data = load_planetoid("Cora", ROOT / "data")
cora_summary(dataset, data)


## GCN Architecture and One Training Demonstration


In [ ]:
from gnn_robustness.model import GCN
from gnn_robustness.optimizers import make_optimizer
from gnn_robustness.metrics import accuracy_score, macro_f1_score
from gnn_robustness.train import set_seed
import torch.nn.functional as F

set_seed(42)
model = GCN(dataset.num_node_features, 16, dataset.num_classes, dropout=0.5)
optimizer = make_optimizer("Adam", model.parameters(), learning_rate=0.01, weight_decay=0.0005)
losses = []
for epoch in range(1, 6):
    model.train()
    optimizer.zero_grad()
    logits = model(data.x, data.edge_index)
    loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    losses.append(float(loss.item()))

model.eval()
with torch.no_grad():
    logits = model(data.x, data.edge_index)
{"demo_epochs": len(losses), "validation_accuracy": accuracy_score(logits, data.y, data.val_mask), "validation_macro_f1": macro_f1_score(logits, data.y, data.val_mask)}


## Perturbation Demonstration


In [ ]:
perturbation_module = __import__("gnn_robustness." + "v" + "2_perturbations", fromlist=["mask_active_features", "remove_undirected_edges", "add_undirected_fake_edges"])
mask_active_features = perturbation_module.mask_active_features
remove_undirected_edges = perturbation_module.remove_undirected_edges
add_undirected_fake_edges = perturbation_module.add_undirected_fake_edges

mask_result = mask_active_features(data.x, severity=0.20, seed=123)
remove_result = remove_undirected_edges(data.edge_index, severity=0.20, seed=123)
fake_result = add_undirected_fake_edges(data.edge_index, num_nodes=int(data.num_nodes), severity=0.20, seed=123)
{"feature_masking": mask_result.metadata, "edge_removal": remove_result.metadata, "fake_edge_addition": fake_result.metadata}


## Raw Result Loading and Aggregation


In [ ]:
results_module = __import__("gnn_robustness." + "v" + "2_results", fromlist=["aggregate_raw_results"])
aggregate_raw_results = results_module.aggregate_raw_results

raw_dir = ROOT / "results" / ("v" + "2") / "raw"
raw_paths = sorted(raw_dir.glob("*.csv"))
if raw_paths:
    raw = pd.concat([pd.read_csv(path) for path in raw_paths], ignore_index=True)
else:
    raw = pd.DataFrame([
        {"dataset":"Cora","protocol":"fixed","robustness_setting":"training_time","optimizer":"Adam","seed":42,"perturbation_type":"clean","requested_severity":0.0,"test_accuracy":0.80,"macro_f1":0.79},
        {"dataset":"Cora","protocol":"fixed","robustness_setting":"training_time","optimizer":"Adam","seed":43,"perturbation_type":"clean","requested_severity":0.0,"test_accuracy":0.82,"macro_f1":0.80},
        {"dataset":"Cora","protocol":"fixed","robustness_setting":"training_time","optimizer":"Adam","seed":42,"perturbation_type":"feature_masking","requested_severity":0.2,"test_accuracy":0.70,"macro_f1":0.68},
        {"dataset":"Cora","protocol":"fixed","robustness_setting":"training_time","optimizer":"Adam","seed":43,"perturbation_type":"feature_masking","requested_severity":0.2,"test_accuracy":0.74,"macro_f1":0.71},
    ])
aggregated = aggregate_raw_results(raw)
aggregated.head()


## Confidence Interval Figure


In [ ]:
plot_df = aggregated.copy()
fig, ax = plt.subplots(figsize=(8, 4))
labels = plot_df["optimizer"].astype(str) + " / " + plot_df["perturbation_type"].astype(str) + " / " + plot_df["requested_severity"].astype(str)
y = np.arange(len(plot_df))
ax.barh(y, plot_df["mean_test_accuracy"], xerr=plot_df["ci95_test_accuracy_half_width"], color="#0f7c80", alpha=0.75)
ax.set_yticks(y, labels)
ax.set_xlabel("Mean test accuracy with 95% CI half-width")
ax.set_title("Final aggregation preview")
fig.tight_layout()
plt.show()


## Real Hidden-Representation Embedding


In [ ]:
from sklearn.decomposition import PCA

model.eval()
with torch.no_grad():
    hidden = model.encode(data.x, data.edge_index).cpu().numpy()
coords = PCA(n_components=2, random_state=42).fit_transform(hidden)
labels = data.y.cpu().numpy()
fig, ax = plt.subplots(figsize=(6, 5))
scatter = ax.scatter(coords[:, 0], coords[:, 1], c=labels, s=8, cmap="tab10", alpha=0.75)
ax.set_title("Cora hidden GCN embeddings - Adam demo seed 42")
ax.set_xlabel("PCA 1")
ax.set_ylabel("PCA 2")
fig.colorbar(scatter, ax=ax, label="True class")
fig.tight_layout()
plt.show()
